# Notebook 1: CogVideoX Generation Assessment

**Purpose:** Run CogVideoX inference on a select few real videos and display input vs. output
frames to verify the first-frame-to-video pipeline is working as intended.

This notebook checks:
- That `extract_first_frame` correctly reads real video files
- That `generate_synthetic_video_cogvideox` produces temporally coherent outputs
- That face identity is roughly preserved across the generated clip

**Steps:**
1. Configure paths and generation parameters
2. Discover source videos on disk
3. Run CogVideoX inference for each video
4. Visualise input vs. generated frames side-by-side
5. Optionally save tensors for use in Notebook 2

## 1. Imports & Configuration

Set `RAW_VIDEO_DIR` to the folder containing your source `.mp4` / `.avi` / `.mov` files.
All other parameters have sensible defaults but can be tuned here:

| Variable | Default | Description |
|---|---|---|
| `NUM_VIDEOS` | `3` | How many source videos to process |
| `GENERATOR_PROMPT` | face description | Text prompt passed to CogVideoX |
| `NUM_INFERENCE_STEPS` | `20` | Denoising steps — higher = better quality, slower |
| `CACHE_DIR` | `None` | Local path to cache model weights (recommended on HPC) |
| `OUTPUT_DIR` | `None` | If set, preprocessed tensors are saved here for Notebook 2 |

In [ ]:
import sys
from pathlib import Path

# Add the project src/ directory to the path so face_fft can be imported
# without requiring a `pip install -e .` step.
_src = Path("../src").resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

In [ ]:
import torch
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from face_fft.data.generate import (
    extract_first_frame,
    generate_synthetic_video_cogvideox,
    preprocess_video_tensor,
)

# ── Configuration ──────────────────────────────────────────────────────────────
RAW_VIDEO_DIR = Path("../src/face_fft/data/raw_videos")  # adjust to your data path
NUM_VIDEOS = 3
GENERATOR_PROMPT = "A highly realistic person's face, talking slightly, photorealistic video."
MODEL_ID = "zai-org/CogVideoX-5b-I2V"
NUM_INFERENCE_STEPS = 20
CACHE_DIR = None          # set to a local path to cache model weights, e.g. Path("/scratch/cache")
OUTPUT_DIR = None         # set to a Path to optionally save .pt tensors, e.g. Path("outputs")

print(f"RAW_VIDEO_DIR : {RAW_VIDEO_DIR.resolve()}")
print(f"NUM_VIDEOS    : {NUM_VIDEOS}")
print(f"DEVICE        : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Source Video Discovery

Recursively search `RAW_VIDEO_DIR` for video files and select the first `NUM_VIDEOS`.
If no files are found, an error is raised — check that `RAW_VIDEO_DIR` is correct.

In [ ]:
VIDEO_EXTENSIONS = [".mp4", ".avi", ".mov"]

video_paths = [
    p
    for ext in VIDEO_EXTENSIONS
    for p in sorted(RAW_VIDEO_DIR.rglob(f"*{ext}"))
]

if not video_paths:
    raise FileNotFoundError(
        f"No video files found under {RAW_VIDEO_DIR}. "
        "Update RAW_VIDEO_DIR in the config cell above."
    )

selected = video_paths[:NUM_VIDEOS]

print(f"Found {len(video_paths)} video(s) total, selected {len(selected)}:")
for p in selected:
    print(f"  {p}")

## 3. CogVideoX Inference Loop

For each selected video, this cell:
1. Extracts the **first frame** as a PIL Image — this is the conditioning input to CogVideoX
2. Calls `generate_synthetic_video_cogvideox` to produce a list of generated PIL frames
3. Converts both real and synthetic sequences into `(C, T, H, W)` float tensors via `preprocess_video_tensor`

> **VRAM note:** Videos are processed one at a time. Each call loads the CogVideoX model,
> runs inference, then moves on. If you hit VRAM limits, reduce `NUM_INFERENCE_STEPS`
> or set `CACHE_DIR` to avoid re-downloading weights on each run.

In [ ]:
results = []  # list of dicts: {path, first_frame, synth_frames, real_tensor, synth_tensor}

for i, video_path in enumerate(selected):
    print(f"\n[{i+1}/{len(selected)}] Processing: {video_path.name}")

    # Extract first frame (PIL Image)
    first_frame = extract_first_frame(str(video_path))
    print(f"  First frame size: {first_frame.size}")

    # Generate synthetic video from the first frame
    print(f"  Generating with CogVideoX ({NUM_INFERENCE_STEPS} steps)...")
    synth_frames = generate_synthetic_video_cogvideox(
        image=first_frame,
        prompt=GENERATOR_PROMPT,
        model_id=MODEL_ID,
        num_inference_steps=NUM_INFERENCE_STEPS,
        cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
    )
    print(f"  Generated {len(synth_frames)} frame(s)")

    # Build tensors for real (single-frame replicated) and synthetic video
    import torchvision.transforms.functional as TF

    def pil_to_tensor(img):
        """Convert PIL Image to float tensor (C, H, W) in [0, 1]."""
        return TF.to_tensor(img)

    # Stack real: repeat first frame to match synth length → (T, H, W, C) → preprocess
    real_tensor_stack = torch.stack(
        [pil_to_tensor(first_frame)] * len(synth_frames), dim=0
    )  # (T, C, H, W)
    # preprocess_video_tensor expects (T, H, W, C) — permute and convert
    real_np = (real_tensor_stack.permute(0, 2, 3, 1).numpy() * 255).astype('uint8')
    real_tensor = preprocess_video_tensor(torch.from_numpy(real_np))

    synth_tensor_stack = torch.stack(
        [pil_to_tensor(f) for f in synth_frames], dim=0
    )  # (T, C, H, W)
    synth_np = (synth_tensor_stack.permute(0, 2, 3, 1).numpy() * 255).astype('uint8')
    synth_tensor = preprocess_video_tensor(torch.from_numpy(synth_np))

    print(f"  real_tensor shape : {real_tensor.shape}")
    print(f"  synth_tensor shape: {synth_tensor.shape}")

    results.append({
        "path": video_path,
        "first_frame": first_frame,
        "synth_frames": synth_frames,
        "real_tensor": real_tensor,
        "synth_tensor": synth_tensor,
    })

print("\nDone. All videos processed.")

## 4. Side-by-Side Visualisation

For each video, a grid is displayed:
- **Column 0:** The real first frame used as the conditioning image
- **Columns 1–4:** Sampled frames from the generated output at indices 0, 4, 8, 12

Use these to assess temporal coherence and face identity preservation before passing
the tensors into the spectral classification pipeline.

In [ ]:
SAMPLE_FRAME_INDICES = [0, 4, 8, 12]  # which synth frames to display

for entry in results:
    n_samples = len(SAMPLE_FRAME_INDICES)
    fig = plt.figure(figsize=(4 * (n_samples + 1), 5))
    gs = gridspec.GridSpec(1, n_samples + 1, wspace=0.05)

    # Column 0: input first frame
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.imshow(entry["first_frame"])
    ax0.set_title("Input\n(first frame)", fontsize=9)
    ax0.axis("off")

    # Columns 1+: sampled generated frames
    for col, fi in enumerate(SAMPLE_FRAME_INDICES):
        if fi >= len(entry["synth_frames"]):
            fi = len(entry["synth_frames"]) - 1
        ax = fig.add_subplot(gs[0, col + 1])
        ax.imshow(entry["synth_frames"][fi])
        ax.set_title(f"Synth\nframe {fi}", fontsize=9)
        ax.axis("off")

    fig.suptitle(entry["path"].name, fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Save Preprocessed Tensors (Optional)

If `OUTPUT_DIR` was set in the config cell, this cell saves each video's real and synthetic
tensors as `.pt` files under `OUTPUT_DIR/real/` and `OUTPUT_DIR/synth/`.

These files can then be loaded directly by `PairedVideoDataset` in **Notebook 2**
(`02_train_and_evaluate.ipynb`) — just point `REAL_DIR` and `SYNTH_DIR` at those subdirectories.

If `OUTPUT_DIR` is `None`, this cell prints a reminder and does nothing.

In [ ]:
if OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(OUTPUT_DIR)
    real_out = OUTPUT_DIR / "real"
    synth_out = OUTPUT_DIR / "synth"
    real_out.mkdir(parents=True, exist_ok=True)
    synth_out.mkdir(parents=True, exist_ok=True)

    for entry in results:
        stem = entry["path"].stem
        torch.save(entry["real_tensor"], real_out / f"{stem}.pt")
        torch.save(entry["synth_tensor"], synth_out / f"{stem}.pt")
        print(f"Saved tensors for {stem}")

    print(f"\nReal tensors  → {real_out}")
    print(f"Synth tensors → {synth_out}")
else:
    print("OUTPUT_DIR is None — tensors not saved. Set OUTPUT_DIR in the config cell to save.")

## What to Look For

When reviewing the side-by-side frames above, check for:

**Motion coherence**
- Does the subject move naturally across frames?
- Are there sudden jumps or flickers between frames 0 → 4 → 8 → 12?

**Face identity consistency**
- Does the face in the generated frames resemble the input first frame?
- CogVideoX is conditioned on the first frame, so strong drift indicates a generation failure.

**Background stability**
- Is the background consistent with the original?
- Large background changes can introduce low-frequency spectral artifacts unrelated to the generation model's compression.

If generation looks good here, the tensors saved in the previous cell can be fed directly
into **Notebook 2** (`02_train_and_evaluate.ipynb`) for spectral analysis and classification.